# Spark SQL Learning 
##  By knowing the process of Read and write, become a  Data INGESTION Developer 
##  By knowing the process of exporting the data, become a  Data Egress Developer  

- Exporting data from databases or applications
- Building APIs that send data to external systems
- Migrating data between cloud platforms
- Managing data transfer pipelines

connecting various sources (files or filesystem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet 
- ORC
- sql
etc....

## Few Facts about Unity Catalog

Unity Catalog is centralized governance solution for managing data, tables, files, machine learning models, permissions, and auditing across all workspaces.


In Unity Catalog, data objects are referenced as Three-Level Namespace


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabular data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if not exists izwd37dev.wd37db;

create volume if not exists izwd37dev.wd37db.rawdatta;

DBFS - Databricks File system
distribuited virtual file system , linux posix format runninng on top of your cloud storages

- /Volumes/catalog/schema/volume-name/path_to_file
- dbfs:/ - uri -> uniform resource identifier (dbricks file system )
- hdfs:/ -> hadoop distruibuited file system file:/ -> local file
- s3a:/ -> aws s3
- gcs:/ -> google storage
- adls:/ -> azure datalake

In [0]:
%fs ls /Volumes/izwd37dev/wd37db/rawdatta/

In [0]:
dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta")


In [0]:
%fs head "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp.csv"

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/sales")

Read data from Spark

Spark session

from pyspark.sql import SparkSession

spark=SparkSession.builder.getOrCreate()

In [0]:
print(spark)

create Dataframe from storage (files / dir ) To read the delimited data from any storage (dbfs , hdfs , lfs , cloud storages )

spark.read.csv option -> create a dataframe

-- csv is the built in source

## Defaults
spark.read.csv

default options :

- header = False

- default cols = _c0 , _c1 _c2

- delimiter = ","

- default data type for all columns = string

In [0]:

empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv") # file path
print(type(empdata)) # dataframe
empdata.show() # similar to collect -> action )
# show action , display default 20 records 
     

In [0]:
empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True) # file path
print(type(empdata)) # dataframe
empdata.show() 
empdata.printSchema()

In [0]:
empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp.csv",header=True,inferSchema=True) # file path
print(type(empdata)) # dataframe
# view few or more records 
empdata.show(5,True)
empdata.printSchema()

In [0]:
display(empdata)

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv") # file path
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()


In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv").toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv",inferSchema=True,header=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()  # here count is 7 because first row considered as header

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv",inferSchema=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()     # header =False so, first row not considered as Header so count is 8


**Different Delimiter**

default is ","

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv",sep="|",inferSchema=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()

spark.read.csv()

-> required input path

-> path could be a file or dir , list of dir ...

In [0]:
empdata_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True)
empdata_df.show(5)

In [0]:
empdata_df=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv","/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv","/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv"],header=True)
empdata_df.show(10)
empdata_df.count()

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv"

**Create sub directory using mkdirs**

In [0]:
%fs mkdirs "/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Chennai"

In [0]:
%fs mkdirs "/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Delhi"

In [0]:
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/",header=True,inferSchema=True,recursiveFileLookup=True)

sales_data_df.show(100)
sales_data_df.printSchema()
sales_data_df.count()  # 1022



In [0]:
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/sales",header=True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter="Sales*")

sales_data_df.show(100)
sales_data_df.printSchema()
sales_data_df.count()   #count :1000




# inferSchema -> generating schema by reading the entire data 
- to avoid reading the data for genrating schema , - performance issue 
- when we are going with inferschema its more dynamic  - data quality issue 


Creating schema in 2 ways.

- using ddl format (available from spark 3x version)
- using programming format (typically using for long days)

In [0]:
creating_schema="empid integer,firstname string,lastname string,profession string,age integer,location string"
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Chennai",header=True,schema=creating_schema)
sales_data_df.show(10)
sales_data_df.printSchema()

In [0]:
creating_schema="branch_id integer,emp_id integer,firstname string,Joining_Date date,lastname string,profession string"
sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=creating_schema,sep="|")
sales_data_df.show(5)
sales_data_df.printSchema()


**create schema using programming method**

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
schema_create=StructType([
    StructField("branch_id",IntegerType()),
    StructField("emp_id",IntegerType()),
    StructField("firstname",StringType()),
    StructField("joining_date",StringType()),
    StructField("lastname",StringType()),
    StructField("profession",StringType())
])

sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=schema_create,sep="|")
sales_data_df.show(5)
sales_data_df.printSchema()      #joining_date is stringtype but we want to convert it into date type so using next approach


**Create schema using programming**
- StructType for row

- StructField for columns

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import to_date
schema_create=StructType([
    StructField("branch_id",IntegerType()),
    StructField("emp_id",IntegerType()),
    StructField("firstname",StringType()),
    StructField("joining_date",StringType()),
    StructField("lastname",StringType()),
    StructField("profession",StringType())
])


sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=schema_create,sep="|")
sales_data_df = sales_data_df.withColumn(
    "joining_date",
    to_date("joining_date", "dd/MM/yyyy")
)
sales_data_df.show(5)
sales_data_df.printSchema()

# **_Read data from JSON_**

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json"


In [0]:

emp_scehma="branch_id integer,emp_id integer,firstname string,lastname string,profession string,age integer, joining_date string,location string"
spark_json_df=spark.read .option("multiline", "true").option("delimiter",",").json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)
spark_json_df.show(5)
spark_json_df.printSchema()


# connect external source -  genric way to read data using spark

- spark.read.option("k","v").format("source").load()
- csv 
- df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp.csv")
 


In [0]:
cust_df=spark.read.option("header","True").option("inferSchema","True").option("delimiter",",").format("csv").load("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv")

cust_df.show()

In [0]:
emp_json_df=spark.read.schema(emp_scehma).format("json").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")

emp_json_df.show()
emp_json_df.printSchema()
     

**create dataframe from any delimited file, comes from any storage**
- local
- hdfs
- dbfs (databricks file system)
- cloud storages (s3,gcs,adls)

df=spark.read.csv()

or

df=spark.read.format("csv").load("url")

inline option --> option("inferschema","True") - passes string value, we can give both "True","TRUE","true"

header =True - passes boolean value (always capital T for True)

In [0]:
df=spark.read.option("inferschema","True").csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True)
df.show()
df.printSchema()